# 🗄️ Advanced SQL Query Generator — Powered by Groq AI

> **A Generative AI project for your resume** | Built with Python · Groq · ipywidgets · Matplotlib

---

## 🚀 5 Key Features

| # | Feature | Description |
|---|---|---|
| 1 | 🧠 **NL → SQL Generation** | Type plain English, get optimized SQL instantly |
| 2 | 💬 **Multi-turn Refinement** | Refine queries conversationally like chatting with an AI |
| 3 | 📊 **Smart Chart Picker** | Auto-selects best chart type from your data |
| 4 | 🔄 **SQL Dialect Converter** | Convert SQL between MySQL, PostgreSQL, SQLite, SQL Server, Oracle |
| 5 | 📁 **CSV → Schema Generator** | Upload CSV → auto-detect types → generate CREATE TABLE SQL |

---
**Get your FREE Groq API key at** 👉 [console.groq.com](https://console.groq.com)

## ⚙️ Cell 1 — Install & Import

In [1]:
%pip install groq ipywidgets matplotlib pandas --quiet
print('✅ All packages installed!')

Note: you may need to restart the kernel to use updated packages.
✅ All packages installed!


In [2]:
import os, json, re, io, csv, textwrap
from datetime import datetime
from collections import defaultdict

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML, clear_output
from groq import Groq

# ── Notebook-wide matplotlib style ──────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0D1117',
    'axes.facecolor':   '#161B22',
    'axes.edgecolor':   '#30363D',
    'axes.labelcolor':  '#E6EDF3',
    'xtick.color':      '#8B949E',
    'ytick.color':      '#8B949E',
    'text.color':       '#E6EDF3',
    'grid.color':       '#21262D',
    'grid.linestyle':   '--',
    'font.family':      'DejaVu Sans',
})

print('✅ Libraries ready!')

✅ Libraries ready!


## 🔑 Cell 2 — Groq API Setup

In [ ]:
# ✏️  Paste your Groq API key below
GROQ_API_KEY = "any_API_key"

# Or load from environment:  GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

client = Groq(api_key=GROQ_API_KEY)
GROQ_MODEL = "model_name"  


print(f'✅ Groq client ready  |  Model: {GROQ_MODEL}')
print(f'🔑 Key preview: {GROQ_API_KEY[:12]}...')

✅ Groq client ready  |  Model: qwen/qwen3.8-27b
🔑 Key preview: gsk_g1cDDxYl...


## 🗃️ Cell 3 — Sample Schemas

In [17]:
SCHEMAS = {
    "🛒 E-Commerce": """
customers  (id INT, name VARCHAR, email VARCHAR, city VARCHAR, created_at DATE)
orders     (id INT, customer_id INT, total DECIMAL, status VARCHAR, created_at DATE)
order_items(id INT, order_id INT, product_id INT, quantity INT, price DECIMAL)
products   (id INT, name VARCHAR, category VARCHAR, price DECIMAL, stock INT)
""".strip(),

    "👥 HR System": """
employees  (id INT, name VARCHAR, department_id INT, salary DECIMAL, hire_date DATE, city VARCHAR)
departments(id INT, name VARCHAR, budget DECIMAL, location VARCHAR)
attendance (id INT, employee_id INT, date DATE, status VARCHAR, hours_worked FLOAT)
payroll    (id INT, employee_id INT, month VARCHAR, gross DECIMAL, deductions DECIMAL, net DECIMAL)
""".strip(),

    "🎓 School": """
students   (id INT, name VARCHAR, grade INT, email VARCHAR, city VARCHAR, enrollment_date DATE)
courses    (id INT, name VARCHAR, credits INT, instructor_id INT, semester VARCHAR)
enrollments(id INT, student_id INT, course_id INT, grade_obtained FLOAT)
instructors(id INT, name VARCHAR, department VARCHAR, email VARCHAR)
""".strip(),
}

ACTIVE_SCHEMA = SCHEMAS["🛒 E-Commerce"]
print("📋 Schemas loaded!")
print("\nActive Schema (E-Commerce):")
print(ACTIVE_SCHEMA)

📋 Schemas loaded!

Active Schema (E-Commerce):
customers  (id INT, name VARCHAR, email VARCHAR, city VARCHAR, created_at DATE)
orders     (id INT, customer_id INT, total DECIMAL, status VARCHAR, created_at DATE)
order_items(id INT, order_id INT, product_id INT, quantity INT, price DECIMAL)
products   (id INT, name VARCHAR, category VARCHAR, price DECIMAL, stock INT)


---
# 🧠 FEATURE 1 — Natural Language → SQL Generation

In [18]:
def generate_sql(question: str, schema: str, model: str = GROQ_MODEL) -> dict:
    """
    Convert a plain-English question into an optimized SQL query.

    Returns dict with keys: success, sql, explanation, breakdown, tips
    """
    system = """You are an expert SQL query generator.
Given a database schema and a natural language question, output ONLY valid JSON:
{
  "sql": "<full SQL query>",
  "explanation": "<one-line summary>",
  "breakdown": ["step1", "step2", "step3"],
  "tips": "<performance tip>"
}
No markdown, no extra text. SQL should work on PostgreSQL/MySQL."""

    user = f"Schema:\n{schema}\n\nQuestion: {question}"

    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "system", "content": system},
                      {"role": "user",   "content": user}],
            max_tokens=1024, temperature=0.1
        )
        raw  = resp.choices[0].message.content.strip()
        raw  = raw.replace("```json", "").replace("```", "").strip()
        data = json.loads(raw)
        return {"success": True, **data}
    except json.JSONDecodeError:
        return {"success": False, "error": "JSON parse error. Try again.", "raw": raw}
    except Exception as e:
        return {"success": False, "error": str(e)}


def show_sql_result(result: dict):
    """Pretty-print a SQL generation result."""
    if not result.get("success"):
        display(Markdown(f"### ❌ Error\n```\n{result.get('error')}\n```"))
        return
    parts = [
        "## ✅ Generated SQL",
        f"```sql\n{result['sql']}\n```",
    ]
    if result.get("explanation"):
        parts.append(f"### 📝 What it does\n{result['explanation']}")
    if result.get("breakdown"):
        parts.append("### 🔍 Breakdown")
        parts += [f"{i+1}. {s}" for i, s in enumerate(result["breakdown"])]
    if result.get("tips"):
        parts.append(f"### 💡 Tip\n> {result['tips']}")
    display(Markdown("\n\n".join(parts)))

print("✅ Feature 1 ready!")

✅ Feature 1 ready!


In [19]:
# ── Feature 1 Interactive Widget ─────────────────────────────────────────────

f1_schema_dd = widgets.Dropdown(
    options=list(SCHEMAS.keys()), description="Schema:",
    layout=widgets.Layout(width="280px")
)
f1_schema_txt = widgets.Textarea(
    value=ACTIVE_SCHEMA, description="", rows=5,
    layout=widgets.Layout(width="98%")
)
f1_schema_dd.observe(lambda c: setattr(f1_schema_txt, 'value', SCHEMAS[c['new']]), names='value')

f1_question = widgets.Textarea(
    value="Show me the top 5 customers by total order value",
    description="Question:", rows=2,
    layout=widgets.Layout(width="98%")
)
f1_btn = widgets.Button(description="⚡ Generate SQL", button_style="success",
                         layout=widgets.Layout(width="180px"))
f1_out = widgets.Output()

def f1_run(b):
    f1_out.clear_output()
    with f1_out:
        print("⏳ Calling Groq AI...")
        res = generate_sql(f1_question.value, f1_schema_txt.value)
        clear_output(wait=True)
        show_sql_result(res)

f1_btn.on_click(f1_run)

display(HTML("<h3>🧠 Feature 1 — Natural Language → SQL</h3>"))
display(f1_schema_dd, f1_schema_txt, f1_question, f1_btn, f1_out)

Dropdown(description='Schema:', layout=Layout(width='280px'), options=('🛒 E-Commerce', '👥 HR System', '🎓 Schoo…

Textarea(value='customers  (id INT, name VARCHAR, email VARCHAR, city VARCHAR, created_at DATE)\norders     (i…

Textarea(value='Show me the top 5 customers by total order value', description='Question:', layout=Layout(widt…

Button(button_style='success', description='⚡ Generate SQL', layout=Layout(width='180px'), style=ButtonStyle()…

Output()

---
# 💬 FEATURE 2 — Multi-turn Conversational Refinement

In [20]:
class SQLConversation:
    """
    Maintains conversation history so every follow-up refinement
    is aware of all previous SQL + questions in the session.
    """
    def __init__(self, schema: str):
        self.schema  = schema
        self.history = []          # list of {role, content} dicts
        self.turns   = []          # list of (question, sql) for display
        self._system = f"""You are an expert SQL assistant with memory of the current conversation.
Database schema:
{schema}

Always respond ONLY with valid JSON (no markdown):
{{"sql": "<query>", "explanation": "<what changed>", "tip": "<optional tip>"}}"""

    def ask(self, question: str) -> dict:
        """Send a question/refinement. Maintains full conversation history."""
        self.history.append({"role": "user", "content": question})
        try:
            resp = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "system", "content": self._system}] + self.history,
                max_tokens=1024, temperature=0.1
            )
            raw = resp.choices[0].message.content.strip()
            raw = raw.replace("```json", "").replace("```", "").strip()
            data = json.loads(raw)
            self.history.append({"role": "assistant", "content": raw})
            self.turns.append((question, data.get("sql", ""), data.get("explanation", "")))
            return {"success": True, **data}
        except Exception as e:
            self.history.pop()   # remove failed user message
            return {"success": False, "error": str(e)}

    def reset(self):
        self.history.clear()
        self.turns.clear()

    def show_history(self):
        """Display the full conversation thread."""
        if not self.turns:
            display(Markdown("_No conversation yet._"))
            return
        lines = ["## 💬 Conversation History"]
        for i, (q, sql, expl) in enumerate(self.turns, 1):
            lines += [
                f"### Turn {i} — {q}",
                f"```sql\n{sql}\n```",
                f"*{expl}*",
            ]
        display(Markdown("\n\n".join(lines)))


# Create a global conversation instance
conv = SQLConversation(ACTIVE_SCHEMA)
print("✅ Feature 2 ready! (Conversation engine loaded)")

✅ Feature 2 ready! (Conversation engine loaded)


In [21]:
# ── Feature 2 Interactive Widget ─────────────────────────────────────────────

f2_schema_dd = widgets.Dropdown(
    options=list(SCHEMAS.keys()), description="Schema:",
    layout=widgets.Layout(width="280px")
)

def f2_schema_change(change):
    global conv
    conv = SQLConversation(SCHEMAS[change['new']])
    f2_out.clear_output()
    with f2_out:
        display(Markdown("_Schema changed. Conversation reset._"))

f2_schema_dd.observe(f2_schema_change, names='value')

f2_input = widgets.Text(
    value="Show top 5 customers by order value",
    description="You:",
    placeholder="Ask or refine...",
    layout=widgets.Layout(width="70%")
)
f2_send  = widgets.Button(description="💬 Send", button_style="success",
                           layout=widgets.Layout(width="110px"))
f2_reset = widgets.Button(description="🔄 Reset", button_style="warning",
                           layout=widgets.Layout(width="110px"))
f2_hist  = widgets.Button(description="📜 History", button_style="info",
                           layout=widgets.Layout(width="110px"))
f2_out   = widgets.Output()

turn_counter = [0]

def f2_send_click(b):
    q = f2_input.value.strip()
    if not q: return
    turn_counter[0] += 1
    with f2_out:
        display(Markdown(f"---\n**🙋 You (Turn {turn_counter[0]}):** {q}"))
        print("⏳ Groq AI thinking...")
    res = conv.ask(q)
    with f2_out:
        if res['success']:
            display(Markdown(
                f"**🤖 AI:** {res.get('explanation', '')}\n\n"
                f"```sql\n{res['sql']}\n```"
                + (f"\n> 💡 {res['tip']}" if res.get('tip') else "")
            ))
        else:
            display(Markdown(f"❌ **Error:** {res['error']}"))

def f2_reset_click(b):
    conv.reset()
    turn_counter[0] = 0
    f2_out.clear_output()
    with f2_out:
        display(Markdown("_Conversation reset. Start fresh!_"))

def f2_hist_click(b):
    f2_out.clear_output()
    with f2_out:
        conv.show_history()

f2_send.on_click(f2_send_click)
f2_reset.on_click(f2_reset_click)
f2_hist.on_click(f2_hist_click)

display(HTML("<h3>💬 Feature 2 — Multi-turn Conversational Refinement</h3>"
             "<p>Generate a query, then refine it step by step in plain English.</p>"))
display(f2_schema_dd)
display(widgets.HBox([f2_input, f2_send, f2_reset, f2_hist]))
display(f2_out)

Dropdown(description='Schema:', layout=Layout(width='280px'), options=('🛒 E-Commerce', '👥 HR System', '🎓 Schoo…

Output()

---
# 📊 FEATURE 3 — Smart Chart Picker (Auto Visualization)

In [22]:
COLORS = ["#58A6FF", "#3FB950", "#F78166", "#D2A8FF", "#FFA657",
          "#79C0FF", "#56D364", "#FF7B72", "#BC8CFF", "#FFB86C"]


def detect_chart_type(df: pd.DataFrame) -> str:
    """
    Automatically pick the best chart type:
      • 1 text col + 1 numeric  → bar
      • date/time col present   → line
      • 1 col with ≤8 unique    → pie
      • 1 text + 2+ numeric     → grouped_bar
      • 2 numeric               → scatter
      • fallback                → bar
    """
    cols      = df.columns.tolist()
    num_cols  = df.select_dtypes(include='number').columns.tolist()
    text_cols = df.select_dtypes(exclude='number').columns.tolist()

    date_like = [c for c in text_cols
                 if any(k in c.lower() for k in ['date','month','year','time','week','day'])]

    if date_like and num_cols:
        return "line"
    if len(text_cols) == 1 and len(num_cols) == 1:
        if df[text_cols[0]].nunique() <= 6:
            return "pie"
        return "bar"
    if len(text_cols) >= 1 and len(num_cols) >= 2:
        return "grouped_bar"
    if len(num_cols) == 2 and len(text_cols) == 0:
        return "scatter"
    return "bar"


def smart_chart(df: pd.DataFrame, title: str = "Query Result"):
    """Auto-pick and render the best chart for the given DataFrame."""
    if df.empty:
        display(Markdown("_No data to visualize._"))
        return

    chart_type = detect_chart_type(df)
    num_cols   = df.select_dtypes(include='number').columns.tolist()
    text_cols  = df.select_dtypes(exclude='number').columns.tolist()

    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor('#0D1117')
    ax.set_facecolor('#161B22')

    label_col  = text_cols[0]  if text_cols  else df.columns[0]
    value_col  = num_cols[0]   if num_cols   else df.columns[-1]

    if chart_type == "bar":
        bars = ax.bar(df[label_col].astype(str), df[value_col],
                      color=COLORS[:len(df)], edgecolor='#30363D', linewidth=0.5)
        ax.bar_label(bars, fmt='%.0f', color='#E6EDF3', padding=3, fontsize=9)
        ax.set_xlabel(label_col, fontsize=11)
        ax.set_ylabel(value_col, fontsize=11)
        plt.xticks(rotation=25, ha='right')
        chart_label = "📊 Bar Chart"

    elif chart_type == "line":
        date_col = text_cols[0]
        for i, nc in enumerate(num_cols):
            ax.plot(df[date_col].astype(str), df[nc],
                    marker='o', color=COLORS[i], linewidth=2.2,
                    label=nc, markersize=5)
        ax.legend(fontsize=9)
        plt.xticks(rotation=35, ha='right')
        ax.set_xlabel(date_col, fontsize=11)
        chart_label = "📈 Line Chart"

    elif chart_type == "pie":
        wedges, texts, autotexts = ax.pie(
            df[value_col], labels=df[label_col].astype(str),
            colors=COLORS[:len(df)], autopct='%1.1f%%',
            pctdistance=0.82, startangle=140,
            wedgeprops={'edgecolor': '#0D1117', 'linewidth': 1.5}
        )
        for t in autotexts: t.set_color('#E6EDF3'); t.set_fontsize(9)
        ax.set_aspect('equal')
        chart_label = "🥧 Pie Chart"

    elif chart_type == "grouped_bar":
        x    = range(len(df))
        w    = 0.8 / len(num_cols)
        for i, nc in enumerate(num_cols):
            offset = [xi + i * w - (len(num_cols)-1)*w/2 for xi in x]
            ax.bar(offset, df[nc], width=w*0.9,
                   color=COLORS[i], label=nc, edgecolor='#30363D')
        ax.set_xticks(list(x))
        ax.set_xticklabels(df[label_col].astype(str), rotation=25, ha='right')
        ax.legend(fontsize=9)
        chart_label = "📊 Grouped Bar Chart"

    elif chart_type == "scatter":
        ax.scatter(df[num_cols[0]], df[num_cols[1]],
                   color=COLORS[0], alpha=0.75, edgecolors='#58A6FF',
                   linewidths=0.5, s=70)
        ax.set_xlabel(num_cols[0], fontsize=11)
        ax.set_ylabel(num_cols[1], fontsize=11)
        chart_label = "🔵 Scatter Plot"

    ax.set_title(f"{chart_label}  —  {title}", fontsize=13, pad=14, color='#E6EDF3')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    display(Markdown(f"*Auto-selected: **{chart_label}** based on your data shape.*"))


print("✅ Feature 3 ready!")

✅ Feature 3 ready!


In [23]:
# ── Feature 3 — Paste CSV data and see the smart chart ───────────────────────

SAMPLE_DATASETS = {
    "Monthly Revenue (→ Line)": """month,revenue,expenses
Jan,42000,28000
Feb,38000,25000
Mar,51000,31000
Apr,63000,35000
May,58000,33000
Jun,72000,40000""",

    "Top Customers (→ Bar)": """customer,total_orders
Alice,142
Bob,98
Charlie,215
Diana,177
Eve,88
Frank,310""",

    "Sales by Category (→ Pie)": """category,sales
Electronics,45000
Clothing,22000
Books,8000
Home,17000
Sports,13000""",

    "Dept Salary & Budget (→ Grouped Bar)": """department,avg_salary,budget
Engineering,95000,500000
Marketing,72000,300000
Sales,65000,280000
HR,60000,200000
Finance,82000,350000""",

    "Price vs Stock (→ Scatter)": """price,stock
12.5,320
45.0,88
8.9,500
120.0,15
65.5,42
29.9,210
85.0,30
5.5,650""",
}

f3_dd = widgets.Dropdown(
    options=list(SAMPLE_DATASETS.keys()),
    description="Dataset:",
    layout=widgets.Layout(width="350px")
)
f3_csv = widgets.Textarea(
    value=list(SAMPLE_DATASETS.values())[0],
    description="CSV Data:",
    rows=8, layout=widgets.Layout(width="98%")
)
f3_title = widgets.Text(
    value="Query Result", description="Chart title:",
    layout=widgets.Layout(width="60%")
)
f3_btn = widgets.Button(description="📊 Draw Smart Chart", button_style="primary",
                         layout=widgets.Layout(width="200px"))
f3_out = widgets.Output()

f3_dd.observe(lambda c: setattr(f3_csv, 'value', SAMPLE_DATASETS[c['new']]), names='value')

def f3_run(b):
    f3_out.clear_output()
    with f3_out:
        try:
            df = pd.read_csv(io.StringIO(f3_csv.value.strip()))
            display(Markdown(f"**Shape:** {df.shape[0]} rows × {df.shape[1]} cols"))
            display(df.head(10))
            smart_chart(df, f3_title.value)
        except Exception as e:
            display(Markdown(f"❌ {e}"))

f3_btn.on_click(f3_run)

display(HTML("<h3>📊 Feature 3 — Smart Chart Picker</h3>"
             "<p>Paste CSV data → AI picks the best chart type automatically.</p>"))
display(f3_dd, f3_csv, f3_title, f3_btn, f3_out)

Dropdown(description='Dataset:', layout=Layout(width='350px'), options=('Monthly Revenue (→ Line)', 'Top Custo…

Textarea(value='month,revenue,expenses\nJan,42000,28000\nFeb,38000,25000\nMar,51000,31000\nApr,63000,35000\nMa…

Text(value='Query Result', description='Chart title:', layout=Layout(width='60%'))

Button(button_style='primary', description='📊 Draw Smart Chart', layout=Layout(width='200px'), style=ButtonSty…

Output()

---
# 🔄 FEATURE 4 — SQL Dialect Converter

In [27]:
DIALECT_NOTES = {
    "MySQL":       "Use DATE_FORMAT(), LIMIT n, IFNULL(), GROUP_CONCAT(), backtick identifiers",
    "PostgreSQL":  "Use TO_CHAR(), LIMIT n, COALESCE(), STRING_AGG(), double-quote identifiers",
    "SQLite":      "Use strftime(), LIMIT n, IFNULL(), GROUP_CONCAT(), minimal type system",
    "SQL Server":  "Use FORMAT(), TOP n, ISNULL(), STRING_AGG(), square-bracket identifiers",
    "Oracle":      "Use TO_CHAR(), ROWNUM/FETCH FIRST n ROWS, NVL(), LISTAGG(), no LIMIT keyword",
}

def convert_sql_dialect(sql: str, from_dialect: str, to_dialect: str) -> dict:
    """
    Convert a SQL query from one dialect to another using Groq AI.
    Handles syntax differences: date functions, LIMIT/TOP/ROWNUM,
    string functions, NULL handling, identifier quoting, etc.
    """
    if from_dialect == to_dialect:
        return {"success": True, "converted_sql": sql,
                "changes": ["Source and target dialect are the same — no changes needed."]}

    system = f"""You are a SQL dialect conversion expert.
Convert the given SQL from {from_dialect} to {to_dialect}.

{to_dialect} syntax notes: {DIALECT_NOTES[to_dialect]}

Respond ONLY with valid JSON (no markdown):
{{"converted_sql": "<converted query>",
  "changes": ["change1", "change2"],
  "warnings": "<any compatibility warnings or empty string>"}}"""

    try:
        resp = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role": "system", "content": system},
                      {"role": "user",   "content": f"Convert this SQL:\n\n{sql}"}],
            max_tokens=1024, temperature=0.1
        )
        raw  = resp.choices[0].message.content.strip()
        raw  = raw.replace("```json", "").replace("```", "").strip()
        data = json.loads(raw)
        return {"success": True, **data}
    except Exception as e:
        return {"success": False, "error": str(e)}


print("✅ Feature 4 ready!")

✅ Feature 4 ready!


In [28]:
# ── Feature 4 Interactive Widget ─────────────────────────────────────────────

DIALECTS = ["MySQL", "PostgreSQL", "SQLite", "SQL Server", "Oracle"]

SAMPLE_SQLS = [
    ("MySQL — date format + LIMIT",
     "SELECT DATE_FORMAT(created_at, '%Y-%m') AS month, SUM(total) AS revenue\nFROM orders\nGROUP BY month\nORDER BY revenue DESC\nLIMIT 5;"),

    ("PostgreSQL — window function",
     "SELECT name, salary,\n  RANK() OVER (PARTITION BY department_id ORDER BY salary DESC) AS rank\nFROM employees;"),

    ("SQL Server — TOP + ISNULL",
     "SELECT TOP 10 name, ISNULL(city, 'Unknown') AS city\nFROM customers\nORDER BY created_at DESC;"),
]

f4_sample_dd = widgets.Dropdown(
    options=[(s[0], s[1]) for s in SAMPLE_SQLS],
    description="Sample:", layout=widgets.Layout(width="400px")
)
f4_sql_in = widgets.Textarea(
    value=SAMPLE_SQLS[0][1], description="Input SQL:",
    rows=7, layout=widgets.Layout(width="98%")
)
f4_sample_dd.observe(lambda c: setattr(f4_sql_in, 'value', c['new']), names='value')

f4_from = widgets.Dropdown(options=DIALECTS, value="MySQL",
                            description="From:", layout=widgets.Layout(width="220px"))
f4_to   = widgets.Dropdown(options=DIALECTS, value="PostgreSQL",
                            description="To:",   layout=widgets.Layout(width="220px"))
f4_btn  = widgets.Button(description="🔄 Convert SQL", button_style="warning",
                          layout=widgets.Layout(width="160px"))
f4_out  = widgets.Output()

def f4_run(b):
    f4_out.clear_output()
    with f4_out:
        print(f"⏳ Converting {f4_from.value} → {f4_to.value}...")
        res = convert_sql_dialect(f4_sql_in.value, f4_from.value, f4_to.value)
        clear_output(wait=True)
        if res['success']:
            parts = [
                f"## ✅ Converted to {f4_to.value}",
                f"```sql\n{res['converted_sql']}\n```",
            ]
            if res.get('changes'):
                parts.append("### 🔄 Changes Made")
                parts += [f"- {c}" for c in res['changes']]
            if res.get('warnings'):
                parts.append(f"### ⚠️ Warnings\n> {res['warnings']}")
            display(Markdown("\n\n".join(parts)))
        else:
            display(Markdown(f"❌ {res['error']}"))

f4_btn.on_click(f4_run)

display(HTML("<h3>🔄 Feature 4 — SQL Dialect Converter</h3>"
             "<p>Convert SQL between MySQL · PostgreSQL · SQLite · SQL Server · Oracle</p>"))
display(f4_sample_dd, f4_sql_in)
display(widgets.HBox([f4_from, f4_to, f4_btn]))
display(f4_out)

Dropdown(description='Sample:', layout=Layout(width='400px'), options=(('MySQL — date format + LIMIT', "SELECT…

Textarea(value="SELECT DATE_FORMAT(created_at, '%Y-%m') AS month, SUM(total) AS revenue\nFROM orders\nGROUP BY…

Output()

---
# 📁 FEATURE 5 — CSV → Schema Generator

In [24]:
def infer_sql_type(series: pd.Series) -> str:
    """Infer SQL column type from a pandas Series."""
    # Try date
    if series.dtype == object:
        try:
            pd.to_datetime(series.dropna().head(20), infer_datetime_format=True)
            if any(k in series.name.lower() for k in ['date','time','at','day','month','year']):
                return "DATE"
        except:
            pass

    dtype = str(series.dtype)
    if 'int'   in dtype: return "INTEGER"
    if 'float' in dtype: return "REAL"
    if 'bool'  in dtype: return "BOOLEAN"

    # Varchar — pick width from max length
    max_len = series.astype(str).str.len().max()
    if max_len <= 50:  return "VARCHAR(100)"
    if max_len <= 255: return "VARCHAR(255)"
    return "TEXT"


def csv_to_schema(csv_text: str, table_name: str = "my_table",
                  dialect: str = "PostgreSQL") -> dict:
    """
    Parse CSV text, infer column types, and generate a CREATE TABLE statement.
    Returns: {table_name, columns: [{name, type, nullable}], sql, stats}
    """
    try:
        df = pd.read_csv(io.StringIO(csv_text.strip()))
    except Exception as e:
        return {"success": False, "error": f"CSV parse error: {e}"}

    # Clean column names
    df.columns = [re.sub(r'[^a-zA-Z0-9_]', '_', c.strip().lower()) for c in df.columns]

    columns = []
    for col in df.columns:
        nullable = df[col].isnull().any()
        sql_type = infer_sql_type(df[col])
        columns.append({"name": col, "type": sql_type, "nullable": nullable})

    # Build CREATE TABLE
    if dialect == "SQL Server":
        pk_type   = "INT IDENTITY(1,1)"
        null_kw   = "NULL" if True else "NOT NULL"
    elif dialect == "Oracle":
        pk_type   = "NUMBER GENERATED ALWAYS AS IDENTITY"
    else:
        pk_type   = "SERIAL" if dialect == "PostgreSQL" else "INTEGER AUTOINCREMENT" if dialect == "SQLite" else "INT AUTO_INCREMENT"

    col_lines = [f"    id          {pk_type} PRIMARY KEY"]
    for col in columns:
        null_part = "" if col["nullable"] else " NOT NULL"
        col_lines.append(f"    {col['name']:<20} {col['type']}{null_part}")

    # dialect-specific timestamp
    ts = "TIMESTAMP DEFAULT CURRENT_TIMESTAMP" if dialect != "Oracle" else "TIMESTAMP DEFAULT SYSDATE"
    col_lines.append(f"    created_at           {ts}")

    sql = f"-- Auto-generated by CSV Schema Generator\n"
    sql += f"-- Dialect: {dialect} | Rows: {len(df)} | Cols: {len(columns)}\n\n"
    sql += f"CREATE TABLE {table_name} (\n"
    sql += ",\n".join(col_lines) + "\n);"

    # Bonus: INSERT template
    col_names  = ", ".join(c['name'] for c in columns)
    placeholders = ", ".join(["%s"] * len(columns))
    insert_sql = f"\n\n-- INSERT template\nINSERT INTO {table_name} ({col_names})\nVALUES ({placeholders});"

    return {
        "success": True,
        "table_name": table_name,
        "columns": columns,
        "sql": sql + insert_sql,
        "stats": {"rows": len(df), "cols": len(columns),
                  "nulls": int(df.isnull().sum().sum())},
        "preview": df.head(5),
    }


print("✅ Feature 5 ready!")

✅ Feature 5 ready!


In [ ]:
# ── Feature 5 Interactive Widget ─────────────────────────────────────────────

SAMPLE_CSVS = {
    "Customer Orders": """customer_name,email,city,total_orders,last_order_date,is_premium
Alice Johnson,alice@mail.com,Hyderabad,14,2024-03-15,True
Bob Smith,bob@mail.com,Mumbai,8,2024-02-28,False
Charlie,charlie@mail.com,,3,2024-01-10,False
Diana Patel,diana@mail.com,Bangalore,22,2024-04-01,True""",

    "Product Inventory": """product_name,category,price,stock_qty,supplier,rating
Laptop Pro,Electronics,75999.99,42,TechCorp,4.5
Cotton Shirt,Clothing,899.00,210,FashionHub,4.1
Python Book,Books,599.50,88,EduPress,4.8
Coffee Maker,Home,3499.00,35,HomeGoods,4.3""",

    "Employee Data": """emp_name,department,salary,hire_date,city,years_exp
Ravi Kumar,Engineering,95000,2020-06-01,Hyderabad,5
Priya Singh,Marketing,72000,2019-03-15,Delhi,7
John Doe,Sales,65000,2021-09-01,Mumbai,3
Anita Rao,HR,,2018-01-20,Hyderabad,9""",
}

f5_dd = widgets.Dropdown(
    options=list(SAMPLE_CSVS.keys()),
    description="Sample CSV:", layout=widgets.Layout(width="300px")
)
f5_csv = widgets.Textarea(
    value=list(SAMPLE_CSVS.values())[0],
    description="CSV Data:", rows=8,
    layout=widgets.Layout(width="98%")
)
f5_table = widgets.Text(
    value="customer_orders", description="Table name:",
    layout=widgets.Layout(width="280px")
)
f5_dialect = widgets.Dropdown(
    options=["PostgreSQL", "MySQL", "SQLite", "SQL Server", "Oracle"],
    value="PostgreSQL", description="Dialect:",
    layout=widgets.Layout(width="220px")
)
f5_btn = widgets.Button(description="📁 Generate Schema", button_style="info",
                         layout=widgets.Layout(width="190px"))
f5_out = widgets.Output()

f5_dd.observe(lambda c: setattr(f5_csv, 'value', SAMPLE_CSVS[c['new']]), names='value')

def f5_run(b):
    f5_out.clear_output()
    with f5_out:
        res = csv_to_schema(f5_csv.value, f5_table.value, f5_dialect.value)
        if not res['success']:
            display(Markdown(f"❌ {res['error']}")); return

        s = res['stats']
        display(Markdown(
            f"### 📊 CSV Stats\n"
            f"- **Rows:** {s['rows']}  |  **Columns:** {s['cols']}  |  **Null cells:** {s['nulls']}"
        ))
        display(Markdown("### 👁️ Preview (first 5 rows)"))
        display(res['preview'])

        display(Markdown("### 🔍 Detected Column Types"))
        type_rows = "".join(
            f"<tr><td><code>{c['name']}</code></td>"
            f"<td><b>{c['type']}</b></td>"
            f"<td>{'✅ nullable' if c['nullable'] else '🔒 not null'}</td></tr>"
            for c in res['columns']
        )
        display(HTML(f"<table><tr><th>Column</th><th>SQL Type</th><th>Null?</th></tr>{type_rows}</table>"))

        display(Markdown(f"### 🏗️ Generated {f5_dialect.value} Schema\n```sql\n{res['sql']}\n```"))

f5_btn.on_click(f5_run)

display(HTML("<h3>📁 Feature 5 — CSV → Schema Generator</h3>"
             "<p>Paste any CSV → auto-detect types → get CREATE TABLE SQL instantly.</p>"))
display(f5_dd, f5_csv)
display(widgets.HBox([f5_table, f5_dialect, f5_btn]))
display(f5_out)

Dropdown(description='Sample CSV:', layout=Layout(width='300px'), options=('Customer Orders', 'Product Invento…

Textarea(value='customer_name,email,city,total_orders,last_order_date,is_premium\nAlice Johnson,alice@mail.com…

Output()